In [24]:
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box as shapely_box
from matplotlib.gridspec import GridSpec
from pyproj import Transformer

# ── 1. 尺寸配置 ──────────────────────────────────────────────────────────────

TOTAL_W_MM = 180  # 增加总宽度以容纳柱状图
TOTAL_H_MM = 70
DPI = 600
W_IN = TOTAL_W_MM / 25.4
H_IN = TOTAL_H_MM / 25.4

# ── 2. 路径配置 ──────────────────────────────────────────────────────────────

HS_PATH = Path("/Volumes/UCL/论文工作/空气污染/HealthServiceIndex.csv")
SHP_PATH = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
SHP_CITY_FIELD = "English"
CHINA_SHP = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国国界线/九段线/九段线和群岛.shp")
OUTFILE = Path("/Users/shirley/Desktop/plots_V2/Fig_HS_distribution.png")

PROJ_STR = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"

# ── 3. 颜色方案（与之前一致）──────────────────────────────────────────────

HS_LEVELS = ["Poor", "Moderate", "Intermediate", "Good", "Excellent", "No data"]
COLOR_SCHEME = {
    "Poor":         "#d7191c",
    "Moderate":     "#f47920",
    "Intermediate": "#f6c101",
    "Good":         "#abd9e9",
    "Excellent":    "#2c7bb6",
    "No data":      "#d3d3d3"
}

# ── 4. 读取HS数据 ────────────────────────────────────────────────────────────

hs_df = pd.read_csv(HS_PATH)
hs_df["city"] = hs_df["city"].str.strip()
hs_df = hs_df[["city", "HS_2", "HS_type2"]].copy()

# ── 5. 读取城市shp ───────────────────────────────────────────────────────────

city_shp = gpd.read_file(SHP_PATH).to_crs(PROJ_STR)
city_shp["city_join"] = city_shp[SHP_CITY_FIELD].str.strip().str.lower()

hs_df["city_join"] = hs_df["city"].str.strip().str.lower()

# ── 6. 合并数据 ──────────────────────────────────────────────────────────────

shp_joined = city_shp.merge(hs_df, on="city_join", how="left")

# 处理缺失值，设置category（基于HS_type2）
shp_joined["category"] = shp_joined["HS_type2"].fillna("No data")
shp_joined["category"] = pd.Categorical(
    shp_joined["category"],
    categories=HS_LEVELS,
    ordered=True
)

# ── 7. 转为点（几何中心）─────────────────────────────────────────────────────

shp_points = shp_joined.copy()
shp_points.geometry = shp_points.geometry.centroid

# ── 8. 胡焕庸线东西分类 ──────────────────────────────────────────────────────

# 胡焕庸线两点（黑河-腾冲）
hhy_transformer = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)
HHY_X, HHY_Y = hhy_transformer.transform(
    [127.5, 98.5],   # 经度
    [50.2,  25.0],   # 纬度
)

hhy_x0, hhy_y0 = HHY_X[0], HHY_Y[0]  # 黑河（北）
hhy_x1, hhy_y1 = HHY_X[1], HHY_Y[1]  # 腾冲（南）

def hhy_x_at_y(y):
    """在胡焕庸线上，给定 y 坐标，返回对应的 x 坐标"""
    t = (y - hhy_y1) / (hhy_y0 - hhy_y1)
    return hhy_x1 + t * (hhy_x0 - hhy_x1)

cx = shp_points.geometry.centroid.x
cy = shp_points.geometry.centroid.y
shp_points["hhy_side"] = np.where(cx > hhy_x_at_y(cy), "East", "West")

# 分离有数据和无数据的城市（基于HS_type2是否为NA）
dat_with_hs = shp_points[~shp_points["HS_type2"].isna()].copy()
dat_no_hs = shp_points[shp_points["HS_type2"].isna()].copy()

print(f"Total cities: {len(shp_points)}")
print(f"Cities with HS_type2: {len(dat_with_hs)}")
print(f"Cities without HS_type2 (No data): {len(dat_no_hs)}")

# ── 9. 读取底图 ──────────────────────────────────────────────────────────────

china_boundary = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
china_boundary2 = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)

# 南海小框边界
NANHAI_BOUNDS = (
    gpd.GeoDataFrame(geometry=[shapely_box(105, 2, 122, 24)], crs="EPSG:4326")
    .to_crs(PROJ_STR)
    .total_bounds
)

# ── 10. 全局样式 ─────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.titlesize": 8,
    "axes.titleweight": "bold",
})

# ── 11. 南海小图函数 ─────────────────────────────────────────────────────────

def add_nanhai_inset(parent_ax):
    """在地图右下角添加九段线南海小框"""
    x1, y1, x2, y2 = NANHAI_BOUNDS
    
    axins = parent_ax.inset_axes([0.75, -0.15, 0.2, 0.3])
    axins.set_facecolor("white")
    
    china_boundary.plot(ax=axins, facecolor="white", edgecolor="black", linewidth=0.2)
    china_boundary2.plot(ax=axins, facecolor="none", edgecolor="black", linewidth=0.4)
    
    if len(dat_no_hs) > 0:
        dat_no_hs.plot(ax=axins, color=COLOR_SCHEME["No data"],
                       markersize=2, alpha=0.5, edgecolor='none', zorder=2)
    
    if len(dat_with_hs) > 0:
        for cat in ["Poor", "Moderate", "Intermediate", "Good", "Excellent"]:
            subset = dat_with_hs[dat_with_hs["category"] == cat]
            if len(subset) > 0:
                subset.plot(ax=axins, color=COLOR_SCHEME[cat],
                           markersize=2, alpha=0.7, edgecolor='none', zorder=3)
    
    axins.set_xlim(x1, x2)
    axins.set_ylim(y1, y2)
    axins.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in axins.spines.values():
        spine.set_linewidth(0.5)
        spine.set_color("black")

# ── 12. 柱状图函数 ───────────────────────────────────────────────────────────

def plot_hhy_bar_chart(ax):
  
    # 统计东西两侧各类别数量
    stats = shp_points.groupby(['hhy_side', 'category']).size().unstack(fill_value=0)
    
    # 确保所有类别都存在
    for cat in HS_LEVELS:
        if cat not in stats.columns:
            stats[cat] = 0
    stats = stats[HS_LEVELS]  # 按顺序排列
    
    # 绘制分组柱状图
    x = np.arange(len(HS_LEVELS))
    width = 0.35
    
    east_counts = stats.loc['East'] if 'East' in stats.index else [0] * len(HS_LEVELS)
    west_counts = stats.loc['West'] if 'West' in stats.index else [0] * len(HS_LEVELS)
    
    # 东侧（浅色）
    bars1 = ax.bar(x - width/2, east_counts, width, 
                   label='East', color='#4db3b3', alpha=0.8, edgecolor='none')
    # 西侧（深色）
    bars2 = ax.bar(x + width/2, west_counts, width,
                   label='West', color='#ca0020', alpha=0.8, edgecolor='none')
    
    # 在柱子上标注数值
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{int(height)}',
                       ha='center', va='bottom', fontsize=5)
    
    # 样式设置
    ax.set_ylabel('Number of cities', fontsize=6)
    ax.set_xticks(x)
    ax.set_xticklabels(HS_LEVELS, rotation=45, ha='right', fontsize=5)
    ax.tick_params(axis='y', labelsize=5)
    ax.legend(fontsize=5, frameon=False, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(0.3)
    ax.grid(axis='y', linewidth=0.2, color='lightgrey', alpha=0.5, zorder=0)
    ax.set_axisbelow(True)
    
    # 标题
    ax.text(-0.15, 1.05, 'b', transform=ax.transAxes,
            fontsize=8, fontweight='bold', va='bottom', ha='left')

# ── 13. 布局：地图 + 柱状图 ──────────────────────────────────────────────────

fig = plt.figure(figsize=(W_IN, H_IN), dpi=DPI)

gs = GridSpec(1, 2, figure=fig, width_ratios=[3, 1], wspace=0.25)

ax_map = fig.add_subplot(gs[0, 0])
ax_bar = fig.add_subplot(gs[0, 1])

# ── 14. 绘制地图 ─────────────────────────────────────────────────────────────

china_boundary.plot(ax=ax_map, facecolor="white", edgecolor="black", linewidth=0.3)
china_boundary2.plot(ax=ax_map, facecolor="none", edgecolor="black", linewidth=0.3)

# 绘制胡焕庸线
ax_map.plot(HHY_X, HHY_Y, color="black", linewidth=0.6,
            linestyle="--", dashes=(4, 3), zorder=5)

# 先画无数据的城市
if len(dat_no_hs) > 0:
    dat_no_hs.plot(ax=ax_map, color=COLOR_SCHEME["No data"],
                   markersize=8, alpha=0.5, edgecolor='none', zorder=2)

# 再画有数据的城市
if len(dat_with_hs) > 0:
    for cat in ["Poor", "Moderate", "Intermediate", "Good", "Excellent"]:
        subset = dat_with_hs[dat_with_hs["category"] == cat]
        if len(subset) > 0:
            subset.plot(ax=ax_map, color=COLOR_SCHEME[cat],
                       markersize=8, alpha=0.7, edgecolor='none', zorder=3, label=cat)

# 添加南海小图
add_nanhai_inset(ax_map)

# 图例
from matplotlib.lines import Line2D
legend_elements = []
for cat in HS_LEVELS:
    if cat == "No data":
        if len(dat_no_hs) > 0:
            legend_elements.append(
                Line2D([0], [0], marker='o', color='w', 
                       markerfacecolor=COLOR_SCHEME[cat], 
                       markersize=4, alpha=0.7, markeredgecolor='none', label=cat)
            )
    else:
        if len(dat_with_hs[dat_with_hs["category"] == cat]) > 0:
            legend_elements.append(
                Line2D([0], [0], marker='o', color='w', 
                       markerfacecolor=COLOR_SCHEME[cat], 
                       markersize=4, alpha=0.7, markeredgecolor='none', label=cat)
            )

ax_map.legend(handles=legend_elements, loc='lower left', fontsize=5,
             frameon=True, facecolor='white', edgecolor='none', framealpha=0.9,
             markerscale=1.2, labelspacing=0.3, handletextpad=0.5)

# 样式调整
ax_map.text(-0.05, 1.05, 'a', transform=ax_map.transAxes,
            fontsize=8, fontweight='bold', va='bottom', ha='left')
ax_map.axis("off")

# 裁剪底部空白
xmin, ymin, xmax, ymax = china_boundary.total_bounds
x_range = xmax - xmin
y_range = ymax - ymin
ax_map.set_xlim(xmin - x_range * 0.01, xmax + x_range * 0.01)
y_bottom = ymin + y_range * 0.15
ax_map.set_ylim(y_bottom, ymax + y_range * 0.01)

# ── 15. 绘制柱状图 ───────────────────────────────────────────────────────────

plot_hhy_bar_chart(ax_bar)

# ── 16. 保存 ─────────────────────────────────────────────────────────────────

plt.tight_layout()
OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"✅ Saved → {OUTFILE}")

Total cities: 370
Cities with HS_type2: 293
Cities without HS_type2 (No data): 77


/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_32499/322994832.py:156: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stats = shp_points.groupby(['hhy_side', 'category']).size().unstack(fill_value=0)
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_32499/322994832.py:278: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


✅ Saved → /Users/shirley/Desktop/plots_V2/Fig_HS_distribution.png
